In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
movies = pd.read_csv("../data/raw/movies.csv")

print("Dataset shape:", movies.shape)

movies.head()

Dataset shape: (9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
print("Columns:")
print(movies.columns)

print("\nMissing values:")
print(movies.isnull().sum())

print("\nDuplicates:")
print(movies.duplicated().sum())

print("\nNumber of movies:", len(movies))

Columns:
Index(['movieId', 'title', 'genres'], dtype='object')

Missing values:
movieId    0
title      0
genres     0
dtype: int64

Duplicates:
0

Number of movies: 9742


In [4]:
# Vérifier les premières valeurs des genres
movies[["title", "genres"]].head(10)

,title,genres
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy
2,Grumpier Old Men (1995),Comedy|Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance
4,Father of the Bride Part II (1995),Comedy
5,Heat (1995),Action|Crime|Thriller
6,Sabrina (1995),Comedy|Romance
7,Tom and Huck (1995),Adventure|Children
8,Sudden Death (1995),Action
9,GoldenEye (1995),Action|Adventure|Thriller


In [5]:
# Remplacer les valeurs manquantes
movies["genres"] = movies["genres"].fillna("")

# Transformer les genres en texte
movies["content_text"] = movies["genres"].str.replace(
    "|",
    " ",
    regex=False
)

# Afficher le résultat
movies[["title", "genres", "content_text"]].head(10)

,title,genres,content_text
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy
5,Heat (1995),Action|Crime|Thriller,Action Crime Thriller
6,Sabrina (1995),Comedy|Romance,Comedy Romance
7,Tom and Huck (1995),Adventure|Children,Adventure Children
8,Sudden Death (1995),Action,Action
9,GoldenEye (1995),Action|Adventure|Thriller,Action Adventure Thriller


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialiser le modèle TF-IDF
tfidf = TfidfVectorizer()

# Transformer les genres en vecteurs numériques
tfidf_matrix = tfidf.fit_transform(movies["content_text"])

# Afficher les dimensions
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

# Afficher les caractéristiques
print("Features:", tfidf.get_feature_names_out())

TF-IDF Matrix Shape: (9742, 24)
Features: ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'genres' 'horror' 'imax'
 'listed' 'musical' 'mystery' 'no' 'noir' 'romance' 'sci' 'thriller' 'war'
 'western']


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculer la similarité entre tous les films
cosine_sim = cosine_similarity(tfidf_matrix)

# Afficher les dimensions
print("Cosine Similarity Shape:", cosine_sim.shape)

Cosine Similarity Shape: (9742, 9742)


In [9]:

def recommend_movies(title, n=10):

    # Rechercher le film par son titre
    title_match = movies[
        movies["title"].str.lower() == title.lower()
    ]

    # Si le titre exact n'existe pas
    if title_match.empty:
        title_match = movies[
            movies["title"].str.contains(
                title,
                case=False,
                regex=False,
                na=False
            )
        ]

    # Vérifier si le film existe
    if title_match.empty:
        print("Film introuvable :", title)
        return pd.DataFrame()

    # Récupérer l'index du film
    movie_idx = title_match.index[0]

    # Calculer les scores de similarité
    similarity_scores = list(
        enumerate(cosine_sim[movie_idx])
    )

    # Trier par similarité décroissante
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclure le film lui-même
    similarity_scores = [
        score for score in similarity_scores
        if score[0] != movie_idx
    ]

    # Garder les N meilleurs films
    top_movies = similarity_scores[:n]

    # Construire le tableau des recommandations
    results = movies.iloc[
        [idx for idx, score in top_movies]
    ][["movieId", "title", "genres"]].copy()

    results["similarity"] = [
        score for idx, score in top_movies
    ]

    return results.reset_index(drop=True)


In [10]:
recommend_movies("Toy Story", 10)

,movieId,title,genres,similarity
0,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
4,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0
5,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
6,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
7,65577,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
8,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
9,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0


In [11]:
recommend_movies("Jumanji", 10)

,movieId,title,genres,similarity
0,60,"Indian in the Cupboard, The (1995)",Adventure|Children|Fantasy,1.0
1,126,"NeverEnding Story III, The (1994)",Adventure|Children|Fantasy,1.0
2,1009,Escape to Witch Mountain (1975),Adventure|Children|Fantasy,1.0
3,2043,Darby O'Gill and the Little People (1959),Adventure|Children|Fantasy,1.0
4,2093,Return to Oz (1985),Adventure|Children|Fantasy,1.0
5,2161,"NeverEnding Story, The (1984)",Adventure|Children|Fantasy,1.0
6,2162,"NeverEnding Story II: The Next Chapter, The (1...",Adventure|Children|Fantasy,1.0
7,2399,Santa Claus: The Movie (1985),Adventure|Children|Fantasy,1.0
8,4896,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure|Children|Fantasy,1.0
9,41566,"Chronicles of Narnia: The Lion, the Witch and ...",Adventure|Children|Fantasy,1.0


In [12]:
recommend_movies("Heat", 10)

,movieId,title,genres,similarity
0,23,Assassins (1995),Action|Crime|Thriller,1.0
1,165,Die Hard: With a Vengeance (1995),Action|Crime|Thriller,1.0
2,185,"Net, The (1995)",Action|Crime|Thriller,1.0
3,288,Natural Born Killers (1994),Action|Crime|Thriller,1.0
4,479,Judgment Night (1993),Action|Crime|Thriller,1.0
5,592,Batman (1989),Action|Crime|Thriller,1.0
6,1036,Die Hard (1988),Action|Crime|Thriller,1.0
7,1752,Hard Rain (1998),Action|Crime|Thriller,1.0
8,1769,"Replacement Killers, The (1998)",Action|Crime|Thriller,1.0
9,1792,U.S. Marshals (1998),Action|Crime|Thriller,1.0
